In [2]:
import pandas as pd
import numpy as np
import sys   
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # add project root so src is importable

from src.config import (
    CUSTOMERS_TRAIN, LOANS_CLEAN, TRANSACTIONS_CLEAN, TRAIN_IDS, CHURN_FEATURES,DEFAULT_FEATURES,SEGMENT_FEATURES,RANDOM_STATE,MODELS_DIR,
) 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score 
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier 

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder  

In [3]:
default_model=pd.read_parquet(DEFAULT_FEATURES)
default_model

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,inflow_to_loan_ratio,defaulted,months_available,total_txns,...,dependents,smartphone_user,complaints_12m,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,age_missing,income_band_missing
0,L500003,C112525,2024-08-15,nano_loan,12857.0,3,0.48,0,2,35.0,...,0,1,1,1,0,0.0,0,444,0,0
1,L500004,C104772,2024-10-22,nano_loan,22631.0,3,0.75,0,4,92.0,...,2,1,0,2,0,0.0,0,465,0,0
2,L500005,C111618,2025-02-25,merchant_advance,158361.0,1,6.09,1,8,103.0,...,0,1,2,0,0,0.0,0,421,0,1
3,L500006,C102207,2025-01-13,nano_loan,21081.0,6,1.33,0,7,28.0,...,1,1,1,1,0,0.0,1,471,0,0
4,L500007,C100980,2025-02-12,nano_loan,11358.0,3,0.26,1,8,33.0,...,1,1,0,4,0,0.0,1,416,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6389,L507994,C108628,2024-08-05,nano_loan,17440.0,1,0.45,0,2,5.0,...,1,1,1,2,1,63900.0,0,483,0,0
6390,L507995,C105831,2025-05-26,device_finance,82169.0,1,2.79,0,11,51.0,...,0,1,0,1,0,0.0,0,426,0,0
6391,L507996,C110319,2025-01-12,merchant_advance,134329.0,1,5.46,0,7,27.0,...,2,1,1,2,0,0.0,0,428,0,0
6392,L507997,C103624,2024-12-31,nano_loan,24820.0,3,0.91,0,6,7.0,...,2,1,0,2,1,27000.0,0,511,0,0


In [4]:
# default_model["defaulted"].value_counts(normalize=True)

In [5]:
X=default_model.drop(columns=["loan_id", "customer_id", "disbursed_date", "defaulted"])
y=default_model["defaulted"] 

In [6]:
# X.dtypes

In [7]:
# X.shape

In [8]:
# X["declared_income_band"].value_counts()

In [9]:
# X["region"].value_counts()

In [10]:
# X["purpose"].value_counts()

In [11]:
# X["declared_income_band"].cat.categories

In [12]:
# X["declared_income_band"] =X["declared_income_band"].cat.codes

In [13]:
# X= pd.get_dummies(X,columns=["region","purpose"],drop_first=True)

In [14]:
cols=["total_txns", "total_value", "average_value_per_mon", "active_months"]
X=X.drop(columns=cols)
X

,purpose,amount_pkr,term_months,inflow_to_loan_ratio,months_available,average_txns_per_mon,active_ratio,age,region,wallet_tenure_months,...,dependents,smartphone_user,complaints_12m,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,age_missing,income_band_missing
0,nano_loan,12857.0,3,0.48,2,17.500000,1.000000,21.0,Punjab,21.0,...,0,1,1,1,0,0.0,0,444,0,0
1,nano_loan,22631.0,3,0.75,4,23.000000,1.000000,34.0,Sindh,21.0,...,2,1,0,2,0,0.0,0,465,0,0
2,merchant_advance,158361.0,1,6.09,8,12.875000,1.000000,23.0,Sindh,19.0,...,0,1,2,0,0,0.0,0,421,0,1
3,nano_loan,21081.0,6,1.33,7,4.000000,0.857143,40.0,Sindh,38.0,...,1,1,1,1,0,0.0,1,471,0,0
4,nano_loan,11358.0,3,0.26,8,4.125000,1.000000,30.0,Punjab,14.0,...,1,1,0,4,0,0.0,1,416,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6389,nano_loan,17440.0,1,0.45,2,2.500000,0.500000,29.0,Islamabad,35.0,...,1,1,1,2,1,63900.0,0,483,0,0
6390,device_finance,82169.0,1,2.79,11,4.636364,0.909091,21.0,Sindh,26.0,...,0,1,0,1,0,0.0,0,426,0,0
6391,merchant_advance,134329.0,1,5.46,7,3.857143,0.857143,32.0,KP,22.0,...,2,1,1,2,0,0.0,0,428,0,0
6392,nano_loan,24820.0,3,0.91,6,1.166667,0.833333,37.0,Balochistan,45.0,...,2,1,0,2,1,27000.0,0,511,0,0


In [15]:
X.shape

(6394, 22)

In [16]:
X.dtypes

purpose                   category
amount_pkr                 float64
term_months                  int64
inflow_to_loan_ratio       float64
months_available             int64
average_txns_per_mon       float64
active_ratio               float64
age                        float64
region                    category
wallet_tenure_months       float64
declared_income_band      category
avg_monthly_inflow_pkr     float64
dependents                   int64
smartphone_user              int64
complaints_12m               int64
failed_txns_12m              int64
has_savings                  int64
savings_balance_pkr        float64
has_insurance                int64
credit_score                 int64
age_missing                  int64
income_band_missing          int64
dtype: object

In [17]:
X_train, X_val ,y_train, y_val=train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE,stratify=y
)

In [18]:
num_cols = X_train.select_dtypes(exclude="category").columns.tolist()
num_cols= ['amount_pkr','term_months','inflow_to_loan_ratio','months_available','average_txns_per_mon','active_ratio','age','wallet_tenure_months','avg_monthly_inflow_pkr','dependents','smartphone_user','complaints_12m','failed_txns_12m','has_savings','savings_balance_pkr','has_insurance','credit_score','age_missing','income_band_missing']

In [19]:
print(X_train.shape)
print(X_val.shape)
print(y_train.mean())
print(y_val.mean())

(5115, 22)
(1279, 22)
0.14115347018572824
0.1415168100078186


In [20]:
# names_columns=X_train.columns

In [21]:
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_val_scaled = scaler.transform(X_val) 

In [22]:
# model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
# model.fit(X_train_scaled, y_train)

In [23]:
# b_pred  = model.predict(X_val_scaled)
# b_proba = model.predict_proba(X_val_scaled)[:, 1]

In [24]:
# print(classification_report(y_val, b_pred))
# print(roc_auc_score(y_val, b_proba)) 

In [25]:
# weights = pd.Series(model.coef_[0], index=X_train.columns)
# weights.sort_values(key=abs, ascending=False)

In [26]:
# X[["total_txns","average_txns_per_mon","total_value"]].corr()

In [27]:
# X[["average_txns_per_mon", "average_value_per_mon", "active_months", "active_ratio", "months_available"]].corr()

RANDOM FOREST

In [28]:
# forest = RandomForestClassifier(
#     n_estimators=300,
#     class_weight="balanced",
#     random_state=RANDOM_STATE,
# )
# forest.fit(X_train, y_train)

# b_pred_tree  = forest.predict(X_val)
# b_proba_tree = forest.predict_proba(X_val)[:, 1]

In [29]:
# print(classification_report(y_val, b_pred_tree))
# print(roc_auc_score(y_val, b_proba_tree)) 

In [30]:
# booster = XGBClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=4,
#     scale_pos_weight=6.1,
#     eval_metric="logloss",
#     early_stopping_rounds=30,
#     random_state=RANDOM_STATE,
# )
# booster.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# b_pred_boost  = booster.predict(X_val)
# b_proba_boost = booster.predict_proba(X_val)[:, 1]

In [31]:
# print(classification_report(y_val, b_pred_boost))
# print(roc_auc_score(y_val, b_proba_boost)) 

In [32]:
# value=(y_train == 0).sum() / (y_train == 1).sum()
# value

In [33]:
# from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# param_grid = {
#     "max_depth": [1,2, 3, 4, 5, 6, 8],
#     "learning_rate": [0.005,0.01, 0.03, 0.05, 0.1, 0.2],
#     "n_estimators": [100, 200, 300, 500, 800],
#     "subsample": [0.6, 0.8, 1.0],
#     "scale_pos_weight": [2, 3, 4, 5]
# }

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# search = RandomizedSearchCV(
#     estimator=XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
#     param_distributions=param_grid,
#     n_iter=30,
#     scoring="average_precision",
#     cv=cv,
#     n_jobs=-1,
#     random_state=RANDOM_STATE,
#     verbose=1,
# )

# search.fit(X_train, y_train)

# print(search.best_params_)
# print(search.best_score_)

In [34]:
# best = search.best_estimator_

# b_pred_tuned  = best.predict(X_val)
# b_proba_tuned = best.predict_proba(X_val)[:, 1]

# print(classification_report(y_val, b_pred_tuned))
# print(roc_auc_score(y_val, b_proba_tuned))

In [35]:
# import numpy as np
# import pandas as pd
# from sklearn.metrics import precision_score, recall_score, f1_score

# # 1. Get raw probabilities for Class 1 (defaulters) using your validation data
# y_probs = search.best_estimator_.predict_proba(X_val)[:, 1]

# # 2. Test different thresholds to see the exact trade-offs
# thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]
# results = []

# for t in thresholds:
#     y_pred_custom = (y_probs >= t).astype(int)
#     results.append({
#         "Threshold": t,
#         "Good Customer Recall (Class 0)": recall_score(y_val, y_pred_custom, pos_label=0),
#         "Defaulter Precision (Class 1)": precision_score(y_val, y_pred_custom, pos_label=1),
#         "Defaulter Recall (Class 1)": recall_score(y_val, y_pred_custom, pos_label=1),
#         "Defaulter F1-Score": f1_score(y_val, y_pred_custom, pos_label=1),
#         "Volume Retained": X_val.loc[y_pred_custom == 0, "amount_pkr"].sum() / X_val["amount_pkr"].sum() 
#     })

# # 3. Print the scannable decision matrix
# print("\n--- Threshold Decision Matrix ---")
# print(pd.DataFrame(results).to_string(index=False))


In [36]:
nominal_cols = ["region", "purpose"]

ordinal_cols = ["declared_income_band"]

num_cols= ['amount_pkr','term_months','inflow_to_loan_ratio','months_available','average_txns_per_mon','active_ratio','age','wallet_tenure_months','avg_monthly_inflow_pkr','dependents','smartphone_user','complaints_12m','failed_txns_12m','has_savings','savings_balance_pkr','has_insurance','credit_score','age_missing','income_band_missing']

income_order = [["<25k", "25-50k", "50-100k", "100-250k", "250k+"]]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("nom", OneHotEncoder(drop="first", handle_unknown="ignore"), nominal_cols),
        ("ord", OrdinalEncoder(categories=income_order), ordinal_cols),
    ],
    remainder="drop",
)

pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])

pipe.fit(X_train, y_train)
b_pred  = pipe.predict(X_val)
b_proba = pipe.predict_proba(X_val)[:, 1]



In [37]:
print(classification_report(y_val, b_pred))
print(roc_auc_score(y_val, b_proba)) 

              precision    recall  f1-score   support

           0       0.93      0.73      0.81      1098
           1       0.29      0.66      0.40       181

    accuracy                           0.72      1279
   macro avg       0.61      0.69      0.61      1279
weighted avg       0.84      0.72      0.76      1279

0.7578973321659673


In [38]:
nominal_cols = ["region", "purpose"]

ordinal_cols = ["declared_income_band"]

num_cols= ['amount_pkr','term_months','inflow_to_loan_ratio','months_available','average_txns_per_mon','active_ratio','age','wallet_tenure_months','avg_monthly_inflow_pkr','dependents','smartphone_user','complaints_12m','failed_txns_12m','has_savings','savings_balance_pkr','has_insurance','credit_score','age_missing','income_band_missing']

income_order = [["<25k", "25-50k", "50-100k", "100-250k", "250k+"]]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("nom", OneHotEncoder(drop="first", handle_unknown="ignore"), nominal_cols),
        ("ord", OrdinalEncoder(categories=income_order), ordinal_cols),
    ],
    remainder="drop",
)

pipe_boost = Pipeline([
    ("prep", preprocess),
    ("model", XGBClassifier(max_depth=1, learning_rate=0.03, n_estimators=200, subsample=0.6, scale_pos_weight=6.08,eval_metric="logloss", random_state=RANDOM_STATE)),
])

pipe_boost.fit(X_train, y_train)
b_pred  = pipe_boost.predict(X_val)
b_proba = pipe_boost.predict_proba(X_val)[:, 1]


In [39]:
print(classification_report(y_val, b_pred))
print(roc_auc_score(y_val, b_proba)) 

              precision    recall  f1-score   support

           0       0.92      0.78      0.84      1098
           1       0.30      0.56      0.39       181

    accuracy                           0.75      1279
   macro avg       0.61      0.67      0.62      1279
weighted avg       0.83      0.75      0.78      1279

0.7727435115579306


In [40]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Get raw probabilities for Class 1 (defaulters) using your validation data
y_probs = pipe.predict_proba(X_val)[:, 1]

# 2. Test different thresholds to see the exact trade-offs
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5,0.6,0.7,0.8,0.9]
results = []

for t in thresholds:
    y_pred_custom = (y_probs >= t).astype(int)
    results.append({
        "Threshold": t,
        "Good Customer Recall (Class 0)": recall_score(y_val, y_pred_custom, pos_label=0),
        "Defaulter Precision (Class 1)": precision_score(y_val, y_pred_custom, pos_label=1),
        "Defaulter Recall (Class 1)": recall_score(y_val, y_pred_custom, pos_label=1),
        "Defaulter F1-Score": f1_score(y_val, y_pred_custom, pos_label=1),
        "Volume Retained": X_val.loc[y_pred_custom == 0, "amount_pkr"].sum() / X_val["amount_pkr"].sum() 
    })

# 3. Print the scannable decision matrix
print("\n--- Threshold Decision Matrix ---")
print(pd.DataFrame(results).to_string(index=False))



--- Threshold Decision Matrix ---
 Threshold  Good Customer Recall (Class 0)  Defaulter Precision (Class 1)  Defaulter Recall (Class 1)  Defaulter F1-Score  Volume Retained
      0.30                        0.352459                       0.188356                    0.911602            0.312204         0.156800
      0.35                        0.448087                       0.202632                    0.850829            0.327311         0.200156
      0.40                        0.541894                       0.221362                    0.790055            0.345828         0.242746
      0.45                        0.633880                       0.247191                    0.729282            0.369231         0.299558
      0.50                        0.725865                       0.285036                    0.662983            0.398671         0.357437
      0.60                        0.846995                       0.348837                    0.497238            0.410023         0

In [41]:
import joblib
bundle = joblib.load(MODELS_DIR / "default_model.joblib")
pipe = bundle["pipeline"]
pipe.named_steps["prep"].get_feature_names_out() 

array(['num__amount_pkr', 'num__term_months', 'num__inflow_to_loan_ratio',
       'num__months_available', 'num__average_txns_per_mon',
       'num__active_ratio', 'num__age', 'num__wallet_tenure_months',
       'num__avg_monthly_inflow_pkr', 'num__dependents',
       'num__smartphone_user', 'num__complaints_12m',
       'num__failed_txns_12m', 'num__has_savings',
       'num__savings_balance_pkr', 'num__has_insurance',
       'num__credit_score', 'num__age_missing',
       'num__income_band_missing', 'nom__region_Balochistan',
       'nom__region_Islamabad', 'nom__region_KP', 'nom__region_Punjab',
       'nom__region_Sindh', 'nom__purpose_emergency',
       'nom__purpose_merchant_advance', 'nom__purpose_nano_loan',
       'ord__declared_income_band'], dtype=object)

In [42]:
"""
RISK BANDS — paste into 06_model_default.ipynb as one cell.

Refits on train only. The joblib artifact was refit on train+test at the end of
train_default.py, so scoring test with it would give optimistic probabilities —
the model has already seen those rows. Band design has to be built on honest
held-out predictions.
"""
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import DEFAULT_FEATURES, DEFAULT_FEATURES_TEST, RANDOM_STATE
from src.models.train_default import make_xy, build_pipeline, NOMINAL, ORDINAL

# ---- rebuild the honest model: train only, test never seen ----
train = pd.read_parquet(DEFAULT_FEATURES)
test = pd.read_parquet(DEFAULT_FEATURES_TEST)

X, y = make_xy(train)
X_test, y_test = make_xy(test)
X_test = X_test[X.columns]

numeric = X.select_dtypes(exclude="category").columns.tolist()
pipe = build_pipeline(numeric)
pipe.fit(X, y)

test_proba = pipe.predict_proba(X_test)[:, 1]

# ---- step 1: look at the distribution before choosing cut points ----
# class_weight="balanced" inflates probabilities, so the spread may not sit
# where you expect. If almost everything lands above 0.3, a "low risk" band
# cut there would be nearly empty and the bands stop being useful.
print("Probability distribution on test:")
print(pd.Series(test_proba).describe().round(3).to_string())
print()

# ---- step 2: assign bands ----
LOW_CUT, HIGH_CUT = 0.3, 0.6

bands = pd.cut(
    test_proba,
    bins=[0, LOW_CUT, HIGH_CUT, 1.0],
    labels=["low", "medium", "high"],
    include_lowest=True,
)

# ---- step 3: does each band actually separate? ----
# The number that matters is the OBSERVED default rate per band. If high shows
# 35% against 5% in low, the bands are real. If all three sit near the 13.6%
# base rate, they are arbitrary slices and should be reported as such.
profile = pd.DataFrame({
    "band": bands,
    "defaulted": y_test.to_numpy(),
    "amount_pkr": test["amount_pkr"].to_numpy(),
})

summary = profile.groupby("band", observed=True).agg(
    loans=("defaulted", "size"),
    default_rate=("defaulted", "mean"),
    volume_pkr=("amount_pkr", "sum"),
)
summary["share_of_loans"] = summary["loans"] / summary["loans"].sum()
summary["share_of_volume"] = summary["volume_pkr"] / summary["volume_pkr"].sum()
summary["lift_vs_base"] = summary["default_rate"] / y_test.mean()

print(f"Base default rate on test: {y_test.mean():.3f}\n")
print(summary.round(3).to_string())

Probability distribution on test:
count    1606.000
mean        0.408
std         0.213
min         0.011
25%         0.256
50%         0.387
75%         0.542
max         0.997

Base default rate on test: 0.136

        loans  default_rate  volume_pkr  share_of_loans  share_of_volume  lift_vs_base
band                                                                                  
low       522         0.033  16248630.0           0.325            0.153         0.239
medium    802         0.138  37885190.0           0.499            0.357         1.015
high      282         0.323  52110721.0           0.176            0.490         2.366


In [43]:
"""
CHURN RISK BANDS — paste into 07_model_churn.ipynb as one cell.

Refits on train only. The joblib artifact was refit on train+test at the end of
train_churn.py, so scoring test with it would give optimistic probabilities —
the model has already seen those rows. Band design has to be built on honest
held-out predictions.

Difference from the default bands: there is no loan amount on a customer, so
value is proxied by avg_monthly_inflow_pkr — how much money flows through the
wallet. That answers "how much is this band worth to keep", which is the
retention equivalent of the volume question.
"""
import pandas as pd

from src.config import CHURN_FEATURES, CHURN_FEATURES_TEST
from src.models.train_churn import make_xy, build_pipeline, NOMINAL, ORDINAL

# ---- rebuild the honest model: train only, test never seen ----
train = pd.read_parquet(CHURN_FEATURES)
test = pd.read_parquet(CHURN_FEATURES_TEST)

X, y = make_xy(train)
X_test, y_test = make_xy(test)
X_test = X_test[X.columns]

numeric = X.select_dtypes(exclude="category").columns.tolist()
pipe = build_pipeline(numeric)
pipe.fit(X, y)

test_proba = pipe.predict_proba(X_test)[:, 1]

# ---- step 1: look at the distribution before choosing cut points ----
# Churn is 7.7% positive against default's 14%, so class_weight="balanced"
# pushes these probabilities harder. The default model's 0.3/0.6 cuts will
# almost certainly be wrong here — read the quartiles below and set the cuts
# from what you actually see, not from what worked on the other model.
print("Probability distribution on test:")
print(pd.Series(test_proba).describe().round(3).to_string())
print()

# ---- step 2: assign bands ----
# Starting point only. Adjust after reading the distribution above: aim for
# bands that are non-trivial in size AND clearly separated on churn rate.
LOW_CUT, HIGH_CUT = 0.4, 0.7

bands = pd.cut(
    test_proba,
    bins=[0, LOW_CUT, HIGH_CUT, 1.0],
    labels=["low", "medium", "high"],
    include_lowest=True,
)

# ---- step 3: does each band actually separate? ----
# The number that matters is the OBSERVED churn rate per band. Separation here
# will be weaker than the default model's 10x spread — average precision was
# 0.183 against a 0.077 base rate, so this is a 2.4x-lift targeting aid, not a
# precise predictor. Report whatever separation is real rather than forcing
# three tiers that do not exist.
profile = pd.DataFrame({
    "band": bands,
    "churned": y_test.to_numpy(),
    "inflow": test["avg_monthly_inflow_pkr"].to_numpy(),
})

summary = profile.groupby("band", observed=True).agg(
    customers=("churned", "size"),
    churn_rate=("churned", "mean"),
    monthly_inflow_pkr=("inflow", "sum"),
)
summary["share_of_customers"] = summary["customers"] / summary["customers"].sum()
summary["share_of_inflow"] = summary["monthly_inflow_pkr"] / summary["monthly_inflow_pkr"].sum()
summary["lift_vs_base"] = summary["churn_rate"] / y_test.mean()

# Churners captured per band — the retention team's coverage question.
summary["share_of_all_churners"] = (
    profile.groupby("band", observed=True)["churned"].sum() / profile["churned"].sum()
)

print(f"Base churn rate on test: {y_test.mean():.3f}\n")
print(summary.round(3).to_string())

Probability distribution on test:
count    2940.000
mean        0.431
std         0.200
min         0.006
25%         0.280
50%         0.445
75%         0.584
max         0.918

Base churn rate on test: 0.077

        customers  churn_rate  monthly_inflow_pkr  share_of_customers  share_of_inflow  lift_vs_base  share_of_all_churners
band                                                                                                                       
low          1249       0.022          53807400.0               0.425            0.471         0.290                  0.123
medium       1426       0.100          51491600.0               0.485            0.451         1.299                  0.630
high          265       0.211           8857300.0               0.090            0.078         2.737                  0.247


In [44]:
"""
SCORING SMOKE TEST — paste into 06_model_default.ipynb as one cell.

Checks three things:
  1. score_default returns the shape the LLM tools expect
  2. the probability matches what the pipeline gives directly
  3. driver labels are human-readable, not raw transformer names
"""
import json
import pandas as pd

from src.config import DEFAULT_FEATURES_TEST, CHURN_FEATURES_TEST
from src.models.scoring import score_default, score_churn

test = pd.read_parquet(DEFAULT_FEATURES_TEST)

# --- 1. an ordinary loan ---
row = test.iloc[[0]]          # double brackets: a one-row FRAME, not a Series
result = score_default(row)
print("LOAN:", row["loan_id"].iloc[0])
print(json.dumps(result, indent=2, default=str))

# --- 2. does the band match the probability? ---
print(f"\nprobability {result['probability']} -> band '{result['band']}'"
      f"   (cuts at 0.30 / 0.60)")

# --- 3. the case that could break _label ---
# One-hot columns arrive as "nom__region_Balochistan". _label splits on the
# base name, so this should render as "region: Balochistan" and not as
# something mangled. Balochistan is the strongest regional coefficient, so it
# should surface as a driver on at least one of these rows.
bal = test[test["region"] == "Balochistan"]
print(f"\n{len(bal)} Balochistan loans in test")
for i in range(min(3, len(bal))):
    drivers = score_default(bal.iloc[[i]])["drivers"]
    print(f"  {bal['loan_id'].iloc[i]}: "
          + " | ".join(f"{d['feature']} = {d['value']} ({d['direction']})"
                       for d in drivers))

# --- 4. band distribution across the whole test set ---
# Should reproduce the risk-band table: roughly 33 / 50 / 18 percent.
bands = [score_default(test.iloc[[i]])["band"] for i in range(len(test))]
print("\nband split:")
print(pd.Series(bands).value_counts(normalize=True).round(3).to_string())

# --- 5. same contract on the churn side ---
churn_test = pd.read_parquet(CHURN_FEATURES_TEST)
c = score_churn(churn_test.iloc[[0]])
print(f"\nCUSTOMER: {churn_test['customer_id'].iloc[0]}")
print(json.dumps(c, indent=2, default=str))

LOAN: L500000
{
  "model": "default",
  "probability": 0.2182,
  "band": "low",
  "drivers": [
    {
      "feature": "declared income band",
      "column": "declared_income_band",
      "value": "100-250k",
      "direction": "decreases risk",
      "contribution": -0.4803
    },
    {
      "feature": "monthly inflow",
      "column": "avg_monthly_inflow_pkr",
      "value": 106100.0,
      "direction": "decreases risk",
      "contribution": -0.2644
    },
    {
      "feature": "loan-to-income ratio",
      "column": "inflow_to_loan_ratio",
      "value": 0.22,
      "direction": "decreases risk",
      "contribution": -0.2211
    }
  ]
}

probability 0.2182 -> band 'low'   (cuts at 0.30 / 0.60)

77 Balochistan loans in test
  L500002: loan-to-income ratio = 21.7 (increases risk) | loan amount = 277761.0 (increases risk) | region: Balochistan = Balochistan (increases risk)
  L500095: region: Balochistan = Balochistan (increases risk) | account age = 17.0 (increases risk) | loan am

In [45]:
from src.models.importance import get_feature_importance
import json

result = get_feature_importance("default")

print("reference levels:", result["reference_levels"])
print()
print("top numeric features:")
for f in result["numeric_features"][:5]:
    print(f"  {f['feature']}: {f['coefficient']} ({f['direction']})")
print()
print("categorical features:")
for f in result["categorical_features"]:
    print(f"  {f['feature']} = {f['level']}: {f['coefficient']} "
          f"(vs {f['compared_to']})")

reference levels: {'region': 'AJK-GB', 'purpose': 'device_finance'}

top numeric features:
  account age: -0.5002 (decreases risk)
  loan-to-income ratio: 0.4207 (increases risk)
  credit score: -0.4114 (decreases risk)
  loan amount: 0.342 (increases risk)
  declared income band: -0.1601 (decreases risk)

categorical features:
  region = Balochistan: 0.5391 (vs AJK-GB)
  loan purpose = emergency: 0.453 (vs device_finance)
  region = Sindh: 0.1712 (vs AJK-GB)
  region = Punjab: 0.1688 (vs AJK-GB)
  region = KP: 0.122 (vs AJK-GB)
  region = Islamabad: 0.1197 (vs AJK-GB)
  loan purpose = merchant_advance: 0.0187 (vs device_finance)
  loan purpose = nano_loan: -0.0139 (vs device_finance)
